# Lambda Azure Engine — Omega Point Runtime
## 200B Parameter Inference on 6GB VRAM via Ternary XOR+POPCNT

This notebook clones the LAE repository from GitHub, runs the full pipeline, and produces a downloadable output artifact.

**Instructions:** Set your GitHub repo URL in Cell 1, then **Runtime → Run All**.

In [ ]:
# ====== CONFIGURATION — EDIT THIS ======
GITHUB_REPO = "https://github.com/teerthsharma/lambda-azure-engine.git"
BRANCH = "main"
# ========================================


## 1. Clone Repository & Install Dependencies

In [ ]:
import os, sys, subprocess

# Clone the repo
if not os.path.exists("/content/lambda-azure-engine"):
    !git clone --branch {BRANCH} --depth 1 {GITHUB_REPO} /content/lambda-azure-engine
else:
    print("Repo already cloned.")

# Add python-bridge to path
BRIDGE_DIR = "/content/lambda-azure-engine/lambda-azure-engine/python-bridge"
if BRIDGE_DIR not in sys.path:
    sys.path.insert(0, BRIDGE_DIR)
os.chdir(BRIDGE_DIR)

# Install dependencies
!pip install -q torch numpy matplotlib safetensors huggingface_hub transformers
try:
    !pip install -q triton
except:
    print("[WARN] Triton install skipped — CPU fallback will be used.")

print("Setup complete.")


## 2. Validate Triton Ternary GEMM Kernel

In [ ]:
from triton_ternary_gemm import test_pack_unpack_roundtrip, test_correctness

test_pack_unpack_roundtrip()
test_correctness(M=64, N=64, K=128)


## 3. KV-Cache Persistent Homology Compression

In [ ]:
import numpy as np
from kv_cache_homology import cluster_and_compress

np.random.seed(42)
keys = np.random.randn(100, 64)
# Inject 20 near-duplicate tokens to show compression
for i in range(20, 40):
    keys[i] = keys[i - 20] + np.random.randn(64) * 0.03

compressed = cluster_and_compress(keys, threshold=0.5)
print(f"Original: {keys.shape[0]} tokens -> Compressed: {compressed.shape[0]} clusters")
print(f"Memory reduction: {(1 - compressed.shape[0]/keys.shape[0])*100:.1f}%")


## 4. Train 150M MoE Model (Quick Validation)

In [ ]:
from train_ternary_moe import TernaryMoEModel, MoEConfig, train
train()


## 5. TWN Quantization → Packed 2-bit $\mathbb{Z}_3$

In [ ]:
from quantize_to_ternary import main as quantize_main
quantize_main()


## 6. (Optional) Stream & Quantize DeepSeek-V2 (236B)

> **Warning:** This cell downloads ~472GB of data, processes it shard-by-shard, and produces a ~55GB packed binary. It will take **several hours** on Colab. Skip this cell if you only want to validate the math.

Uncomment the last line to run.

In [ ]:
from stream_and_quantize_deepseek import stream_and_quantize

stream_and_quantize(model_id="deepseek-ai/DeepSeek-V2", output_file="200b_lae_ternary_packed.bin")


## 7. Evaluate Perplexity

In [ ]:
from evaluate_perplexity import evaluate
evaluate()


## 8. Benchmark Suite (Perplexity vs VRAM)

In [ ]:
from benchmark_suite import main as bench_main
bench_main()


## 9. Automated LLM Testing

In [ ]:
from integration_test import run_tests
run_tests()


## 10. Download Output Artifacts

In [ ]:
import os, zipfile, shutil

OUTPUT_DIR = "/content/lae_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Collect all generated artifacts
artifacts = [
    "moe_fp16.pt",
    "moe_ternary_packed.bin",
    "benchmark.png",
    "200b_lae_ternary_packed.bin",
    "test_report.txt",
]

for f in artifacts:
    if os.path.exists(f):
        shutil.copy(f, os.path.join(OUTPUT_DIR, f))
        print(f"  Collected: {f} ({os.path.getsize(f)/1e6:.1f} MB)")
    else:
        print(f"  Skipped (not found): {f}")

# Create downloadable zip
zip_path = "/content/lae_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            filepath = os.path.join(root, file)
            zf.write(filepath, os.path.relpath(filepath, OUTPUT_DIR))

print(f"\nArtifact archive: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")

# Trigger browser download
try:
    from google.colab import files
    files.download(zip_path)
    print("Download triggered.")
except ImportError:
    print("Not running in Colab — file saved to", zip_path)
